# GPR Simulation Workflow

Full pipeline in one notebook:

| Step | Cell | What it does |
|------|------|--------------|
| 0 | Config | Set all paths and parameters |
| 1 | Write inputs | Create `.poly` and FEM input files |
| 2 | TetGen | Mesh the geometry |
| 3 | Solver | Run `elfe3d_gpr` |
| 4 | Load results | Read `electric_fields_receiver_line.txt` |
| 5 | Plot | Compare with analytical reference |

In [1]:
# ── Standard imports ────────────────────────────────────────────────────────
import os, sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# ── Project imports (all four files must be in the same directory) ──────────
from workflow               import GPRWorkflow, WorkflowConfig
from outputs.fieldreader    import AnalyticalLoader
from outputs.visualize      import ErrorStatPlot

ModuleNotFoundError: No module named 'fieldreader'

---
## ⚙️  Step 0 — Configuration
Edit the values below. Everything else in the notebook stays fixed.

In [ ]:
cfg = WorkflowConfig(

    # ── Paths ────────────────────────────────────────────────────────────────
    base_dir      = r"C:\Projects\GPR",    # project root (Windows path)
    run_name      = "in_air_only",          # input sub-folder  →  base_dir/run_name/
    exec_rel_path = "../elfe3d_gpr",        # relative to base_dir  ("../../solver/elfe3d_gpr" etc.)
    poly_filename = "GPR_model.poly",       # written by write_inputs(), read by TetGen

    # ── TetGen ───────────────────────────────────────────────────────────────
    tetgen_bin    = "tetgen15",
    tetgen_flags  = "-pq1.2kAaen",

    # ── Environment ──────────────────────────────────────────────────────────
    use_wsl       = True,   # True  = running notebook from Windows, needs WSL
                            # False = already inside WSL

    # ── Receiver layout (must match your survey setup) ────────────────────────
    num_endfire   = 64,
    num_broadside = 0,
    num_oblique   = 0,

    # ── FEM model parameters (passed straight to write_inputs) ────────────────
    # Add any keyword arguments that your writeinputfiles / writetetgenpoly
    # functions expect.  Example:
    model_params = dict(
        frequency   = 100e6,       # Hz
        background_er = 1.0,
        # ... add your own parameters here
    ),
)

wf = GPRWorkflow(cfg)

print(f"Input dir : {wf.input_dir}")
print(f"Poly file : {wf.poly_path}")
print(f"Executable: {wf.exec_path}")

---
## 📝  Step 1 — Write input files
Creates the `.poly` geometry and all FEM input files inside `base_dir/run_name/`.

In [ ]:
wf.write_inputs()

# Verify the files exist
for f in sorted(os.listdir(wf.input_dir)):
    print(f"  {f}")

---
## 🔺  Step 2 — Run TetGen
Meshes the `.poly` geometry.  Mesh files are written into the same folder.

In [ ]:
wf.run_tetgen()

# Show the mesh files that TetGen produced
mesh_exts = {".node", ".ele", ".face", ".edge", ".neigh"}
for f in sorted(os.listdir(wf.input_dir)):
    if any(f.endswith(ext) for ext in mesh_exts):
        print(f"  {f}")

---
## ⚡  Step 3 — Run the FEM solver
Calls `./elfe3d_gpr` from its own directory.  This can take a while.

In [ ]:
wf.run_solver()

---
## 📂  Step 4 — Load results

In [ ]:
results = wf.load_results(label="elfe3D")

ef = results[0]   # Endfire  (always present)
# bs = results[1] # Broadside (if num_broadside > 0)
# ob = results[2] # Oblique   (if num_oblique  > 0)

print(f"Endfire receivers: {len(ef.r)}")
print(f"r range: {ef.r.min():.2f} – {ef.r.max():.2f} m")

---
## 📊  Step 5 — Load analytical reference and plot
Edit the CSV path below to point at your Evert / empymod reference file.

In [ ]:
# ── Load analytical reference ────────────────────────────────────────────────
ANALYTICAL_CSV = r"C:\Projects\GPR\semi-analytic_100MHz\Exx_single_freq_4_100MHz.csv"

evert = AnalyticalLoader(ANALYTICAL_CSV, label="Evert").endfire()

In [ ]:
# ── 2×2 comparison ───────────────────────────────────────────────────────────
wf.plot_comparison(
    analytical = evert,
    suptitle   = "Endfire – Comparison with Analytical Solution",
)

In [ ]:
# ── 2×2 error plot ───────────────────────────────────────────────────────────
wf.plot_errors(
    analytical = evert,
    suptitle   = "Endfire – Errors vs Analytical Solution",
)

In [ ]:
# ── 2×4 combined (fields + errors) ──────────────────────────────────────────
wf.plot_combined(
    analytical = evert,
    suptitle   = "Endfire – Fields and Errors",
)

In [ ]:
# ── Error histograms ─────────────────────────────────────────────────────────
wf.plot_histograms(
    analytical = evert,
    suptitle   = "Error Distribution – Endfire",
)

In [ ]:
# ── Printed error summary ────────────────────────────────────────────────────
wf.print_error_summary(analytical=evert)

---
## 🔁  Parameter sweep (optional)
Run multiple configurations (e.g. different PML thicknesses) and compare them
in a single `ErrorStatPlot`.

In [ ]:
# Example: sweep over PML thickness
# Each entry is (run_name, label, pml_thickness_value)
sweep = [
    ("run_pml_10",   r"$\lambda/10$",   0.30),
    ("run_pml_15",   r"$\lambda/15$",   0.20),
    ("run_pml_20",   r"$\lambda/20$",   0.15),
]

sweep_results = []
for run_name, label, pml_val in sweep:
    run_cfg = WorkflowConfig(
        **{**cfg.__dict__,          # copy base config
           "run_name": run_name,
           "model_params": {**cfg.model_params, "pml_thickness": pml_val},
        }
    )
    run_wf = GPRWorkflow(run_cfg)

    # run_wf.run_all()   # uncomment to actually run each case

    ds = run_wf.load_results(label=label)
    sweep_results.append(ds[0])   # endfire only

# Error stat plot
ErrorStatPlot(
    datasets     = sweep_results,
    reference    = evert,
    param_values = [v for _, _, v in sweep],
    xtick_labels = [lbl for _, lbl, _ in sweep],
).plot(suptitle="Error Statistics vs PML Thickness")